# 05 — Results visualization: watching the regime sequence

Turns the fitted rSLDS into animations, so the **certified** result — *decision-making traverses a reliable, position-dependent sequence of metastable regimes* — is something you can watch rather than read off a table.

## ⚠️ The one scientific guardrail

The certified rSLDS result is the **sequence** (recurrence test + `order_consistency`, both backends, both alignments — `docs/worklog/2026-07-15_rslds-first-real-test.md`). The **geometry** of each regime (attractor vs line-attractor/integrator) is **not** certified: the `|λ|`-stability test showed `|λ|≈1` is >97 % reproduced on *time-shuffled* data, i.e. it is the smoother, not the dynamics.

So every animation here shows **when the population switches regime and in what order**. The latent plane is drawn only so the switches have somewhere to happen — *its shape carries no claim*, and the panels say so. The load-bearing content is the **regime tape** (single trial) and the **switch-time density** (occupancy sweep).

## Two visualizations, and why both

| | `single_trial_gif` | `occupancy_sweep_gif` |
|---|---|---|
| what moves | a marker walking **one trial's** latent path, flashing on each switch | a time cursor sweeping the **trial-averaged** occupancy of all 3 areas |
| the payload | the **regime tape** (3 lanes, ▼ at each switch) | the **switch-time density** (when, on average, each area switches) |
| honest about | *one* decision, as it happens — vivid but a single noisy sample | the *average* — nothing rests on one path; this is the certified quantity |
| trust it for | intuition | "which area switches when" |

We start with the **single example trial** (the intuition), then the **occupancy sweep** (the trustworthy version). One figure of each, for **both alignments** (`stimOn` and `firstMovement`) — because the `[0,+0.8] s` window is mostly *movement* (median RT 171 ms), so a sequence that looks stimulus-locked may be movement-locked. Disagreement between the two alignments is information.

The animation code lives in `src/viz/rslds_anim.py` (reusable); this notebook just orchestrates.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data.preprocess import WORKING_EID, load_processed
from src.dynamics.rslds import counts_to_latents, fit_rslds
from src.viz.rslds_anim import (
    pick_representative_trial, single_trial_gif, occupancy_sweep_gif, modal_sequence, _collapse,
)
from src.viz.style import set_style
set_style()

REGIONS = ["VISp", "SCm", "MRN"]
K_PER_REGION = {"VISp": 3, "SCm": 4, "MRN": 3}
N_LATENT, SIGMA_MS, BIN_S, STICKY, SEED = 3, 40.0, 0.04, 2.0, 0
OUT = ROOT / "results" / "figures" / "phase5_viz"

# The two frozen 40 ms tensors (built by 00_preprocessing.ipynb / build_processed_40ms.py).
ALIGN = {
    "stimOn":        (f"{WORKING_EID[:8]}__stimOn_pre400_post800_bin40_rate1.npz",        "stimulus onset"),
    "firstMovement": (f"{WORKING_EID[:8]}__firstMovement_pre400_post800_bin40_rate1.npz", "first movement"),
}
print(f"repo: {ROOT}")

## Fit the rSLDS once per alignment

Fast exact-EM backend, `K = 3/4/3` from the HMM stability analysis, 40 ms / σ = 40 ms (the ground-truth-validated configuration). ~1.5 min per alignment. These are the same fits Part 10 of `03_rslds_triplet.ipynb` visualises statically; here we animate them.

In [ ]:
def fit_all(counts_by_region):
    fits = {}
    for r in REGIONS:
        x, _, _ = counts_to_latents(counts_by_region[r], n_latent=N_LATENT,
                                    smooth_sigma_ms=SIGMA_MS, bin_size_s=BIN_S)
        fits[r] = fit_rslds(x, K_PER_REGION[r], recurrent=True, n_iter=60,
                            n_restarts=3, seed=SEED, sticky=STICKY)
    return fits

data, fits = {}, {}
for key, (fname, _) in ALIGN.items():
    t0 = time.time()
    d = load_processed(ROOT / "data" / "processed" / fname)
    data[key] = d
    fits[key] = fit_all({r: d["regions"][r]["counts"] for r in REGIONS})
    print(f"{key:14s} {d['n_trials']} trials, {len(d['bin_centers'])} bins  "
          f"[fit {time.time()-t0:.0f}s]")

## 1 — Single example trial (the intuition)

We pick a trial that **follows each area's modal regime sequence** and, among those, **shows a post-onset switch in as many areas as possible** (so the illustration isn't a trial where a weak-sequence area like VISp happens to sit still). The bonus is small — it selects an illustrative trial *among the representative ones*, it does not manufacture a sequence.

Read the GIF **bottom-up**: the **regime tape** is the result (3 lanes, coloured by regime, ▼ at each switch, orange cursor = now). The latent panels on top are the same information embodied as motion, with the honest caption that their *shape* is a smoothing artefact.

In [ ]:
for key, (_, label) in ALIGN.items():
    d, f = data[key], fits[key]
    bc, rt = d["bin_centers"], d["trials"].get("rt")
    tr = pick_representative_trial(f, rt=rt, bin_centers=bc)

    print(f"\n=== {key} — trial #{tr} ===")
    for r in REGIONS:
        print(f"   {r:5s} modal seq {modal_sequence(f[r].states)} | "
              f"this trial {_collapse(f[r].states[tr])}")

    t0 = time.time()
    gif = single_trial_gif(f, tr, bc, OUT / f"single_trial_{key}.gif",
                           align_label=label, fps=8)
    print(f"   saved {gif.relative_to(ROOT)}  ({gif.stat().st_size//1024} KB) "
          f"[{time.time()-t0:.0f}s]")
    display(Image(filename=str(gif)))

## 2 — The occupancy sweep (the trustworthy version)

The single trial is vivid but it is **one sample**, and a single-trial regime path is noisy — VISp especially (the ground truth recovers only ~71 % of its regimes even at 40 ms). The occupancy sweep removes that fragility by showing **averages over all trials**, and it is the visualization to trust for *"which area switches when"*.

**What sweeps, and why it is the right object.** Three panels stacked (VISp / SCm / MRN) each show `P(regime | t)` — the fraction of trials in each regime at each moment, i.e. the trial-averaged occupancy from `03`'s Part 10. A single time cursor crosses all three **together**, so you read the areas *in register*: at any instant you see which regime dominates each area simultaneously (they share the trial clock because they were recorded simultaneously).

**The payload panel — switch-time density.** Beneath the occupancies, one curve per area: the mean number of regime *switches per trial* entering each time bin. This is the certified content made visual:

- a **peak** = a moment when that area reliably switches regime across trials;
- the **order of the peaks** across areas *is* the cross-area sequence — the direction-agnostic cascade — without ever committing to attractor geometry;
- the **area under each curve** = its mean switches per trial (its `switch_rate`), so a broad low curve (VISp) vs a sharp early peak (SCm) is visible at a glance.

Why this is honest where the single trial is only suggestive: occupancy and switch-time density are **statistics over every trial**, so nothing rests on one noisy path, and neither uses the flow matrix `A` (whose eigenvalues are the artefact). It is the animated form of exactly the quantities the recurrence test certified.

**How to read the two alignments together.** If a switch-density peak sits at a *fixed latency* under `stimOn` but *smears* under `firstMovement` (or vice-versa), that tells you whether the switch is locked to the stimulus or to the movement — the movement-window question, answered visually. (The parallel HMM analysis found the first evoked transition is largely movement-locked; watch whether the rSLDS switch peaks agree.)

In [ ]:
for key, (_, label) in ALIGN.items():
    d, f = data[key], fits[key]
    t0 = time.time()
    gif = occupancy_sweep_gif(f, d["bin_centers"], OUT / f"occupancy_sweep_{key}.gif",
                              align_label=label, fps=8)
    print(f"{key}: saved {gif.relative_to(ROOT)}  ({gif.stat().st_size//1024} KB) "
          f"[{time.time()-t0:.0f}s]")
    display(Image(filename=str(gif)))

## What to take from these

- **The sequence is real and you can watch it** — regimes are entered in a reproducible order, strongest in SCm (4 regimes) and MRN, weakest in VISp (which often sits in one regime — honestly shown, not hidden).
- **The switch-time density is the cross-area cascade**, direction-agnostic, and it is the certified content — no attractor-geometry claim is made or needed.
- **The latent-plane shapes are illustration only** (smoothing artefact), and every panel says so.
- **Next:** regenerate on the 5-session pool for replication, and read the `stimOn` vs `firstMovement` switch-density peaks side by side to see how much of the cascade is movement-locked.

GIFs are written to `results/figures/phase5_viz/`. The notebook ships with outputs **cleared** (project rule: distrust committed notebook outputs) — re-run to regenerate.